# Ring Attention 教程

本教程介绍 Ring Attention 分布式长序列注意力的核心概念和实现。

## 目录
1. 在线 Softmax 算法
2. 分块注意力计算
3. Ring Attention 机制
4. 序列并行

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, 'src')

from ring_attention import (
    RingAttentionConfig,
    BlockwiseAttention,
    RingAttention,
    SequenceParallel,
    compute_blockwise_attention,
    create_ring_attention,
)

## 1. 在线 Softmax 算法

Ring Attention 的核心是在线 softmax，支持分块累积计算。

In [ ]:
# 标准 softmax vs 在线 softmax
def standard_softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def online_softmax_demo(blocks):
    """演示在线 softmax 分块计算"""
    m = -np.inf  # 最大值
    d = 0.0      # exp 和
    
    for block in blocks:
        m_new = max(m, np.max(block))
        d = d * np.exp(m - m_new) + np.sum(np.exp(block - m_new))
        m = m_new
    
    # 最终计算
    result = np.concatenate([np.exp(b - m) / d for b in blocks])
    return result

# 测试
x = np.random.randn(16)
blocks = [x[:4], x[4:8], x[8:12], x[12:]]

standard = standard_softmax(x)
online = online_softmax_demo(blocks)

print(f"标准 softmax: {standard[:4]}")
print(f"在线 softmax: {online[:4]}")
print(f"误差: {np.max(np.abs(standard - online)):.2e}")

## 2. 分块注意力计算

In [ ]:
# 使用 BlockwiseAttention
attn = BlockwiseAttention(head_dim=32, causal=True)

batch, n_heads, seq_len, head_dim = 2, 4, 16, 32
q = np.random.randn(batch, n_heads, seq_len, head_dim)
k = np.random.randn(batch, n_heads, seq_len, head_dim)
v = np.random.randn(batch, n_heads, seq_len, head_dim)

max_score, sum_exp, out = attn(q, k, v)
result = attn.finalize(out, sum_exp)

print(f"Input: {q.shape}")
print(f"Output: {result.shape}")

## 3. Ring Attention 机制

In [ ]:
# 创建 Ring Attention
config = RingAttentionConfig(
    d_model=64,
    n_heads=4,
    block_size=16,
    n_devices=4,
)

print(f"模型维度: {config.d_model}")
print(f"设备数量: {config.n_devices}")
print(f"每设备序列长度: {config.block_size}")
print(f"最大序列长度: {config.max_seq_len}")

In [ ]:
# Ring Attention 前向传播
ring_attn = RingAttention(config)
x = np.random.randn(2, 16, 64)  # [batch, block_size, d_model]

# 模拟不同设备
y0 = ring_attn(x, device_id=0)
y1 = ring_attn(x, device_id=1)

print(f"Device 0 output: {y0.shape}")
print(f"Device 1 output: {y1.shape}")

## 4. 序列并行

In [ ]:
# 序列分割与收集
sp = SequenceParallel(config)

# 长序列
long_seq = np.random.randn(2, 60, 64)
print(f"原始序列: {long_seq.shape}")

# 分割到 4 个设备
blocks = sp.split_sequence(long_seq)
print(f"分割后: {len(blocks)} 块, 每块 {blocks[0].shape}")

# 收集回来
recovered = sp.gather_sequence(blocks, original_len=60)
print(f"恢复后: {recovered.shape}")
print(f"数据一致: {np.allclose(recovered, long_seq)}")

In [ ]:
# 使用工厂函数
attn = create_ring_attention(
    d_model=512,
    n_heads=8,
    block_size=1024,
    n_devices=8,
)
print(f"Max sequence length: {attn.config.max_seq_len}")

## 总结

Ring Attention 核心优势:
1. **分布式计算**: 序列分布到多设备
2. **内存高效**: 每设备只需 O(L/P) 内存
3. **近乎无限序列**: 支持百万级 token